# Metaclasses em Python

**Metaclasse** é a "classe das classes" — define como uma classe se comporta durante sua criação.

Em Python, toda classe é, por padrão, uma instância de `type`. Ao criar uma metaclasse personalizada (herdando de `type`), é possível interceptar e modificar o processo de criação de qualquer classe que a utilize.

**Casos de uso comuns:**
- Injetar atributos ou métodos automaticamente em todas as classes de um sistema
- Validar a estrutura das classes em tempo de definição
- Construir frameworks com comportamento padronizado (ex.: Django ORM, SQLAlchemy)

**Métodos principais de uma metaclasse:**

| Método | Quando é chamado |
|---|---|
| `__new__` | Antes de a classe ser criada — permite modificar seu dicionário |
| `__init__` | Após a classe ser criada — permite configurações adicionais |
| `__call__` | Quando a classe é chamada para criar uma instância |

## Exemplo 1 — Injeção de atributos via metaclasse

A metaclasse `MeuMeta` intercepta a criação de qualquer classe que a use e adiciona automaticamente o atributo `novo_atributo` ao seu dicionário, sem que a classe precise declará-lo.

In [57]:
class MeuMeta(type):
    """Metaclasse que injeta um atributo de classe em toda classe que a utilizar."""

    def __new__(cls, nome, bases, dct):
        # Adiciona o atributo antes de a classe ser construída
        dct['novo_atributo'] = "Valor adicionado pela metaclasse"
        return super().__new__(cls, nome, bases, dct)

In [58]:
class MinhaClasse(metaclass=MeuMeta):
    """Classe vazia que recebe `novo_atributo` automaticamente via MeuMeta."""
    pass

In [59]:
# `novo_atributo` não foi declarado em MinhaClasse — foi injetado pela metaclasse
obj = MinhaClasse()
print(obj.novo_atributo)  # type: ignore[attr-defined]

Valor adicionado pela metaclasse


In [60]:
class ValidadorMeta(type):
    """
    Metaclasse que gera métodos set_<atributo> com validação de tipo.

    A classe filha deve declarar um atributo `validacoes` contendo uma lista
    de tuplas (nome_atributo, tipo_esperado). Para cada par, um método
    `set_<nome_atributo>(self, value)` é criado e adicionado à classe.
    """

    def __new__(cls, nome, bases, dct):
        validacoes = dct.get('validacoes', [])

        for attr, tipo in validacoes:
            if not callable(tipo):
                raise TypeError(
                    f"Tipo de validação para '{attr}' deve ser uma função ou classe."
                )

            def valida_func(self, value, attr=attr, tipo=tipo):
                """Valida o tipo e atribui o valor ao atributo correspondente."""
                if not isinstance(value, tipo):
                    raise TypeError(
                        f"Atributo '{attr}' deve ser do tipo {tipo.__name__}."
                    )
                setattr(self, attr, value)

            # Nomeia o método dinamicamente e o registra no dicionário da classe
            valida_func.__name__ = f"set_{attr}"
            dct[f"set_{attr}"] = valida_func

        return super().__new__(cls, nome, bases, dct)

## Exemplo 2 — Metaclasse com validação de tipo

`ValidadorMeta` lê uma lista `validacoes` declarada na classe e gera automaticamente métodos `set_<atributo>` que validam o tipo antes de atribuir o valor.

**Fluxo:**
1. A classe define `validacoes = [('atributo', Tipo), ...]`
2. `ValidadorMeta.__new__` itera essa lista e cria um método `set_<atributo>` para cada par
3. Cada método gerado verifica o tipo do valor recebido e lança `TypeError` se inválido, ou atribui com `setattr` se válido

In [61]:
class Usuario(metaclass=ValidadorMeta):
    """
    Modelo de usuário com validação automática de tipos via ValidadorMeta.

    Atributos gerados automaticamente pela metaclasse:
        set_nome(value: str)   — aceita apenas str
        set_idade(value: int)  — aceita apenas int
    """

    validacoes = [
        ('nome', str),
        ('idade', int),
    ]

    def __init__(self, nome, idade):
        self.set_nome(nome)    # type: ignore[attr-defined]
        self.set_idade(idade)  # type: ignore[attr-defined]

In [62]:
# Criação válida: tipos corretos
try:
    user = Usuario("Brunno", 30)
    print(f"Usuário criado: {user.nome}, {user.idade}")  # type: ignore[attr-defined]

    # Tentativa de atribuir um valor com tipo incorreto (int em vez de str)
    user.set_nome(123)  # type: ignore[attr-defined]
except TypeError as e:
    print(f"Erro: {e}")

Usuário criado: Brunno, 30
Erro: Atributo 'nome' deve ser do tipo str.
